# Task 1 of 3 (`Land_New_Data`)
### Lakeflow Jobs Orchestration Lab · CDC + Medallion Architecture
**Databricks Free Edition (serverless)**

This notebook is the **first task** in a multi-task Job. Its only responsibility is to
**land a new batch from the CDC feed** into the Volume (drop a JSON file into
`landing/`).

> **Why we simulate the feed:** In production, a CDC feed is generated by Debezium or AWS DMS
> by reading the source database transaction log. In Free Edition, **we do not provision that
> infrastructure**; instead, we drop JSON files into the Volume. Each line represents an event
> (`INSERT` / `UPDATE` / `DELETE`) with a `sequence_num` that defines the event order.

**Orchestration idea:** every time the Job runs, this task drops a
**different batch** so the pipeline always has new data to process. The batch is selected
using a **Job parameter** (`batch`), which defaults to `1`.

In [0]:
# Shared configuration (identical across the 3 Job tasks)
catalog = "workspace"
schema  = "medallion_dbsql"
volume  = "raw_data"

base_path = f"/Volumes/{catalog}/{schema}/{volume}"
landing   = f"{base_path}/landing"     # CDC feed events are "landed" here

# Ensure the required structure exists (idempotent)
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {catalog}.{schema}.{volume}")
spark.sql(f"USE {catalog}.{schema}")

print("catalog/schema :", f"{catalog}.{schema}")
print("landing        :", landing)

catalog/schema : workspace.medallion_dbsql
landing        : /Volumes/workspace/medallion_dbsql/raw_data/landing


## Job Parameter: Which Batch to Land

Lakeflow Jobs allows you to pass **parameters** to each task. Here, we read a parameter
named `batch`. If the Job does not provide it (for example, if you run the notebook manually),
we use `1` as the default.

`dbutils.widgets` is the standard mechanism for receiving Job parameters.

In [0]:
# Read the "batch" parameter passed by the Job (default = "1")
dbutils.widgets.text("batch", "1")     # Create the widget / default value
batch = dbutils.widgets.get("batch")

# After creating the "JobArrival"
# batch = '3'
print(f"Batch requested by the Job: {batch}")

Batch requested by the Job: 1


## Helper Function to Drop Files into the Volume

In serverless, we can write directly to `/Volumes/...` using Python's `open()`.

In [0]:
import os

def drop_file(folder, filename, content):
    os.makedirs(folder, exist_ok=True)
    path = f"{folder}/{filename}"
    with open(path, "w") as f:
        f.write(content)
    print(f"  + file created: {path}")

print("Helper drop_file() is ready")

Helper drop_file() is ready


## Defining the CDC Feed Batches

We define three possible batches. The Job determines which one to land using the `batch` parameter:

- **Batch 1** — 3 new orders (all `INSERT`)
- **Batch 2** — 1 `UPDATE`, 1 `DELETE`, and 1 `INSERT` (affecting orders that already existed)
- **Batch 3** — 2 additional `UPDATE` operations and 1 new `INSERT`

Notice how `sequence_num` continues increasing across batches (1,2,3 → 4,5,6 → 7,8,9). This allows the
pipeline to determine which changes occurred **later**.

In [0]:
batches = {
    "1": [
        '{"order_id":1,"customer":"Ana","country":"Colombia","city":"Bogota","category":"Footwear","product":"Sneakers","quantity":1,"unit_price":90.0,"amount":90.0,"status":"pending","order_date":"2026-07-01","operation":"INSERT","sequence_num":1}',
        '{"order_id":2,"customer":"Beto","country":"Mexico","city":"Mexico City","category":"Clothing","product":"T-Shirt","quantity":2,"unit_price":25.0,"amount":50.0,"status":"pending","order_date":"2026-07-01","operation":"INSERT","sequence_num":2}',
        '{"order_id":3,"customer":"Carla","country":"Argentina","city":"Buenos Aires","category":"Accessories","product":"Cap","quantity":1,"unit_price":15.0,"amount":15.0,"status":"pending","order_date":"2026-07-02","operation":"INSERT","sequence_num":3}',
        '{"order_id":4,"customer":"Diego","country":"Chile","city":"Santiago","category":"Clothing","product":"Jeans","quantity":1,"unit_price":65.0,"amount":65.0,"status":"pending","order_date":"2026-07-02","operation":"INSERT","sequence_num":4}',
        '{"order_id":5,"customer":"Elena","country":"Peru","city":"Lima","category":"Outerwear","product":"Jacket","quantity":1,"unit_price":120.0,"amount":120.0,"status":"pending","order_date":"2026-07-03","operation":"INSERT","sequence_num":5}',
        '{"order_id":6,"customer":"Fabio","country":"Ecuador","city":"Quito","category":"Bags","product":"Backpack","quantity":1,"unit_price":55.0,"amount":55.0,"status":"pending","order_date":"2026-07-03","operation":"INSERT","sequence_num":6}',
        '{"order_id":7,"customer":"Gabriela","country":"Spain","city":"Madrid","category":"Accessories","product":"Watch","quantity":1,"unit_price":210.0,"amount":210.0,"status":"pending","order_date":"2026-07-04","operation":"INSERT","sequence_num":7}',
        '{"order_id":8,"customer":"Hugo","country":"United States","city":"Miami","category":"Accessories","product":"Sunglasses","quantity":1,"unit_price":80.0,"amount":80.0,"status":"pending","order_date":"2026-07-04","operation":"INSERT","sequence_num":8}',
        '{"order_id":9,"customer":"Isabel","country":"Brazil","city":"Sao Paulo","category":"Clothing","product":"Sweater","quantity":2,"unit_price":45.0,"amount":90.0,"status":"pending","order_date":"2026-07-05","operation":"INSERT","sequence_num":9}',
        '{"order_id":10,"customer":"Javier","country":"Uruguay","city":"Montevideo","category":"Footwear","product":"Boots","quantity":1,"unit_price":140.0,"amount":140.0,"status":"pending","order_date":"2026-07-05","operation":"INSERT","sequence_num":10}',
        '{"order_id":11,"customer":"Karen","country":"Costa Rica","city":"San Jose","category":"Accessories","product":"Scarf","quantity":3,"unit_price":20.0,"amount":60.0,"status":"pending","order_date":"2026-07-06","operation":"INSERT","sequence_num":11}',
        '{"order_id":12,"customer":"Luis","country":"Panama","city":"Panama City","category":"Accessories","product":"Belt","quantity":2,"unit_price":30.0,"amount":60.0,"status":"pending","order_date":"2026-07-06","operation":"INSERT","sequence_num":12}',
        '{"order_id":13,"customer":"Marta","country":"Canada","city":"Toronto","category":"Clothing","product":"Dress","quantity":1,"unit_price":95.0,"amount":95.0,"status":"pending","order_date":"2026-07-07","operation":"INSERT","sequence_num":13}',
        '{"order_id":14,"customer":"Nicolas","country":"Germany","city":"Berlin","category":"Accessories","product":"Hat","quantity":2,"unit_price":18.0,"amount":36.0,"status":"pending","order_date":"2026-07-07","operation":"INSERT","sequence_num":14}',
        '{"order_id":15,"customer":"Olga","country":"France","city":"Paris","category":"Footwear","product":"Sandals","quantity":1,"unit_price":45.0,"amount":45.0,"status":"pending","order_date":"2026-07-08","operation":"INSERT","sequence_num":15}',
        '{"order_id":16,"customer":"Pablo","country":"Japan","city":"Tokyo","category":"Accessories","product":"Wallet","quantity":1,"unit_price":60.0,"amount":60.0,"status":"pending","order_date":"2026-07-08","operation":"INSERT","sequence_num":16}',
        '{"order_id":17,"customer":"Raul","country":"Australia","city":"Sydney","category":"Outerwear","product":"Coat","quantity":1,"unit_price":180.0,"amount":180.0,"status":"pending","order_date":"2026-07-09","operation":"INSERT","sequence_num":17}',
        '{"order_id":18,"customer":"Sofia","country":"United Kingdom","city":"London","category":"Bags","product":"Handbag","quantity":1,"unit_price":110.0,"amount":110.0,"status":"pending","order_date":"2026-07-09","operation":"INSERT","sequence_num":18}'
    ],
    "2": [
        '{"order_id":1,"customer":"Ana","country":"Colombia","city":"Bogota","category":"Footwear","product":"Sneakers","quantity":1,"unit_price":90.0,"amount":90.0,"status":"shipped","order_date":"2026-07-01","operation":"UPDATE","sequence_num":19}',
        '{"order_id":2,"customer":"Beto","country":"Mexico","city":"Mexico City","category":"Clothing","product":"T-Shirt","quantity":2,"unit_price":25.0,"amount":50.0,"status":"canceled","order_date":"2026-07-01","operation":"DELETE","sequence_num":20}',
        '{"order_id":5,"customer":"Elena","country":"Peru","city":"Lima","category":"Outerwear","product":"Jacket","quantity":1,"unit_price":120.0,"amount":120.0,"status":"shipped","order_date":"2026-07-03","operation":"UPDATE","sequence_num":21}',
        '{"order_id":7,"customer":"Gabriela","country":"Spain","city":"Madrid","category":"Accessories","product":"Watch","quantity":1,"unit_price":210.0,"amount":210.0,"status":"shipped","order_date":"2026-07-04","operation":"UPDATE","sequence_num":22}',
        '{"order_id":9,"customer":"Isabel","country":"Brazil","city":"Sao Paulo","category":"Clothing","product":"Sweater","quantity":2,"unit_price":45.0,"amount":90.0,"status":"canceled","order_date":"2026-07-05","operation":"DELETE","sequence_num":23}',
        '{"order_id":13,"customer":"Marta","country":"Canada","city":"Toronto","category":"Clothing","product":"Dress","quantity":1,"unit_price":95.0,"amount":95.0,"status":"shipped","order_date":"2026-07-07","operation":"UPDATE","sequence_num":24}',
        '{"order_id":16,"customer":"Pablo","country":"Japan","city":"Tokyo","category":"Accessories","product":"Wallet","quantity":1,"unit_price":60.0,"amount":60.0,"status":"canceled","order_date":"2026-07-08","operation":"DELETE","sequence_num":25}',
        '{"order_id":19,"customer":"Victor","country":"Italy","city":"Rome","category":"Electronics","product":"Headphones","quantity":1,"unit_price":150.0,"amount":150.0,"status":"pending","order_date":"2026-07-10","operation":"INSERT","sequence_num":26}',
        '{"order_id":20,"customer":"Wendy","country":"Netherlands","city":"Amsterdam","category":"Electronics","product":"Keyboard","quantity":1,"unit_price":95.0,"amount":95.0,"status":"pending","order_date":"2026-07-10","operation":"INSERT","sequence_num":27}',
        '{"order_id":21,"customer":"Xavier","country":"Switzerland","city":"Zurich","category":"Electronics","product":"Monitor","quantity":2,"unit_price":220.0,"amount":440.0,"status":"pending","order_date":"2026-07-11","operation":"INSERT","sequence_num":28}',
        '{"order_id":22,"customer":"Yolanda","country":"South Korea","city":"Seoul","category":"Footwear","product":"Running Shoes","quantity":1,"unit_price":130.0,"amount":130.0,"status":"pending","order_date":"2026-07-11","operation":"INSERT","sequence_num":29}'
    ],
}

if batch not in batches:
    raise ValueError(f"Batch '{batch}' is not defined. Use '1', '2', or '3'.")

content = "\n".join(batches[batch])
drop_file(landing, f"cdc_batch_{int(batch):02d}.json", content)
print(f"\nBatch {batch} landed in the landing directory ({len(batches[batch])} events).")

  + file created: /Volumes/workspace/medallion_dbsql/raw_data/landing/cdc_batch_01.json

Batch 1 landed in the landing directory (18 events).


## Verification: What's on the landing page now?

In [0]:
# List the feed files that exist so far
for f in dbutils.fs.ls(landing):
    print(f.name, "-", f.size, "bytes")

cdc_batch_01.json - 4370 bytes


---
**Task 1 completed.** The new CDC feed has been landed in the Volume.
**Task 2** (`Process_Medallion`) will read from this landing area and apply the CDC changes
through the Bronze → Silver → Gold layers.

> In the Job, this task has **no dependencies**; it is the starting point of the DAG.